# K-Nearest Neighbours (KNN), Explained Simply
Predict by looking at the most similar examples. Runs free in Google Colab.

Run each cell from top to bottom with **Shift + Enter**.

## 1. The data
A clothing shop knows the height, weight and chosen T-shirt size (S, M or L) of 240 customers.

### 240 customers
- `tshirt_sizes.csv`: 240 customers: height in metres, weight in kilograms, and size

In [ ]:
import pandas as pd

url = ("https://raw.githubusercontent.com/Uttam-Nepal/"
       "machine-learning-explained/main/"
       "k-nearest-neighbours/tshirt_sizes.csv")
df = pd.read_csv(url)
print(df.shape)
print(df["size"].value_counts().to_dict())
df.head()

### Plot the customers
- `df.groupby("size")`: one group per size, so each gets its own colour
- `the picture`: S bottom left, L top right, M in between, with some overlap

In [ ]:
import matplotlib.pyplot as plt

colors = {"S": "tab:blue", "M": "tab:orange", "L": "tab:green"}
for s, part in df.groupby("size"):
    plt.scatter(part["height_m"], part["weight_kg"],
                c=colors[s], label=s, alpha=0.7)
plt.xlabel("height (m)"); plt.ylabel("weight (kg)")
plt.legend(); plt.show()

## 2. Distance
KNN measures similarity with the straight-line (Euclidean) distance: **square root of (difference in x squared + difference in y squared)**.

### Distance by hand and with NumPy
- `(6 - 3)**2 + (5 - 1)**2`: 9 + 16 = 25
- `np.sqrt`: the square root: 5
- `np.linalg.norm(a - b)`: the same distance in one step

In [ ]:
import numpy as np

a = np.array([3, 1])
b = np.array([6, 5])
print("by hand:", np.sqrt((6 - 3)**2 + (5 - 1)**2))
print("NumPy:  ", np.linalg.norm(a - b))

## 3. KNN in code
To predict a new customer: find the **k** nearest customers, and take a majority vote of their sizes. Scale the inputs first, so both count fairly.

### Without and with scaling
- `KNeighborsClassifier(n_neighbors=5)`: KNN with k = 5
- `fit`: KNN just stores the training customers; there's nothing else to learn
- `without scaling`: heights vary by about 0.1 m, weights by about 13 kg: weight dominates the distance, height is almost ignored
- `with scaling`: both inputs count fairly: better accuracy

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X = df[["height_m", "weight_kg"]]
y = df["size"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0, stratify=y)
raw = KNeighborsClassifier(n_neighbors=5).fit(X_train, y_train)
knn = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))
knn.fit(X_train, y_train)
print("without scaling:", round(raw.score(X_test, y_test), 3))
print("with scaling:   ", round(knn.score(X_test, y_test), 3))

### A new customer and their neighbours
- `knn.predict(new)`: the majority vote of the 5 nearest customers
- `kneighbors`: which training customers are the nearest (after scaling)
- `the table`: the 5 neighbours and their sizes

In [ ]:
new = pd.DataFrame({"height_m": [1.75], "weight_kg": [72]})
print("predicted size:", knn.predict(new)[0])
dist, idx = knn[-1].kneighbors(knn[0].transform(new))
print(X_train.iloc[idx[0]].assign(size=y_train.iloc[idx[0]].values))

## 4. Choosing k
k = 1 copies the single nearest customer (memorising). A huge k averages over almost everyone (too simple). Pick k with cross-validation.

### Cross-validation accuracy for each k
- `ks`: eight values of k to try
- `cross_val_score`: 5-fold cross-validation on the training customers only
- `the output`: k = 1 is worst; around 15 is best; very large k gets worse again

In [ ]:
from sklearn.model_selection import cross_val_score

ks = [1, 3, 5, 9, 15, 25, 51, 101]
scores = []
for k in ks:
    model = make_pipeline(StandardScaler(), KNeighborsClassifier(k))
    scores.append(cross_val_score(model, X_train, y_train, cv=5).mean())
for k, s in zip(ks, scores):
    print(f"k = {k:>3}: {s:.3f}")
plt.plot(ks, scores, marker="o")
plt.xscale("log"); plt.xlabel("k"); plt.ylabel("cross-validation accuracy")
plt.show()

### What k does to the boundaries
- `np.meshgrid`: a fine grid of heights and weights covering the chart
- `m.predict(grid)`: the size KNN would recommend at every point
- `contourf`: colour each region by its predicted size
- `k = 1`: jagged islands around single customers: overfitting
- `k = 15`: smooth, sensible regions

In [ ]:
xx, yy = np.meshgrid(np.linspace(1.45, 2.0, 200),
                     np.linspace(35, 115, 200))
grid = pd.DataFrame({"height_m": xx.ravel(), "weight_kg": yy.ravel()})
num = {"S": 0, "M": 1, "L": 2}
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, k in zip(axes, [1, 15]):
    m = make_pipeline(StandardScaler(), KNeighborsClassifier(k))
    m.fit(X_train, y_train)
    zz = pd.Series(m.predict(grid)).map(num).values.reshape(xx.shape)
    ax.contourf(xx, yy, zz, alpha=0.3, cmap="brg")
    ax.scatter(X_train["height_m"], X_train["weight_kg"],
               c=y_train.map(num), cmap="brg", s=12)
    ax.set_title(f"k = {k}")
plt.show()

## 5. Mini project: a size recommender
Recommend a size, with how sure KNN is.

### Size recommender
- `KNeighborsClassifier(15)`: the k chosen by cross-validation
- `predict_proba`: the share of the 15 neighbours voting for each size
- `final.classes_`: the size names, in the same order as the probabilities

In [ ]:
final = make_pipeline(StandardScaler(), KNeighborsClassifier(15))
final.fit(X_train, y_train)
print("test accuracy:", round(final.score(X_test, y_test), 3))
for h, w in [(1.60, 52), (1.75, 72), (1.88, 95)]:
    person = pd.DataFrame({"height_m": [h], "weight_kg": [w]})
    votes = final.predict_proba(person)[0].round(2)
    probs = {c: float(v) for c, v in zip(final.classes_, votes)}
    print(f"{h} m, {w} kg -> {final.predict(person)[0]}  votes: {probs}")

## Try it yourself
1. Recommend a size for someone 1.62 m and 80 kg.
2. Try weights='distance' in KNeighborsClassifier, so closer neighbours count more. Does accuracy change?
3. Work out the distance between (1, 2) and (4, 6) by hand, then check with np.linalg.norm.